In [0]:
from pyspark.sql import functions as F

inventory_bronze = spark.table(
    "workspace.fashion_bronze.inventory_ingested"
)

print("Bronze inventory records:", inventory_bronze.count())
inventory_bronze.printSchema()

Bronze inventory records: 40000
root
 |-- product_id: integer (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- inventory_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- inventory_value: double (nullable = true)



In [0]:
inventory_bronze.select(
    F.count("*").alias("total_records"),
    F.countDistinct("product_id").alias("unique_products"),
    F.countDistinct("warehouse_id").alias("warehouses"),
    F.sum(F.col("product_id").isNull().cast("int")).alias("null_product_ids"),
    F.sum(F.col("warehouse_id").isNull().cast("int")).alias("null_warehouse_ids"),
    F.min("stock_quantity").alias("minimum_stock"),
    F.max("stock_quantity").alias("maximum_stock"),
    F.sum("inventory_value").alias("total_inventory_value")
).show()

+-------------+---------------+----------+----------------+------------------+-------------+-------------+---------------------+
|total_records|unique_products|warehouses|null_product_ids|null_warehouse_ids|minimum_stock|maximum_stock|total_inventory_value|
+-------------+---------------+----------+----------------+------------------+-------------+-------------+---------------------+
|        40000|          10000|         4|               0|                 0|            0|          500| 3.111003714305003...|
+-------------+---------------+----------+----------------+------------------+-------------+-------------+---------------------+



In [0]:
print("INVENTORY SCHEMA")
inventory_bronze.printSchema()

print("WAREHOUSE DISTRIBUTION")
inventory_bronze.groupBy("warehouse_id") \
    .count() \
    .orderBy("warehouse_id") \
    .show()

print("INVENTORY STATUS")
inventory_bronze.groupBy("inventory_status") \
    .count() \
    .orderBy("inventory_status") \
    .show()

INVENTORY SCHEMA
root
 |-- product_id: integer (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- inventory_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- inventory_value: double (nullable = true)

WAREHOUSE DISTRIBUTION
+------------+-----+
|warehouse_id|count|
+------------+-----+
|       WH001|10000|
|       WH002|10000|
|       WH003|10000|
|       WH004|10000|
+------------+-----+

INVENTORY STATUS
+----------------+-----+
|inventory_status|count|
+----------------+-----+
|        In Stock|35054|
|       Low Stock| 4868|
|    Out of Stock|   78|
+----------------+-----+



In [0]:
print("Negative stock quantities:",
      inventory_bronze.filter(F.col("stock_quantity") < 0).count())

print("Negative reorder levels:",
      inventory_bronze.filter(F.col("reorder_level") < 0).count())

print("Invalid inventory values:",
      inventory_bronze.filter(F.col("inventory_value") < 0).count())

print("Inventory status mismatches:",
      inventory_bronze.filter(
          ((F.col("stock_quantity") == 0) & (F.col("inventory_status") != "Out of Stock")) |
          ((F.col("stock_quantity") > 0) & (F.col("stock_quantity") <= F.col("reorder_level")) & (F.col("inventory_status") != "Low Stock")) |
          ((F.col("stock_quantity") > F.col("reorder_level")) & (F.col("inventory_status") != "In Stock"))
      ).count())

Negative stock quantities: 0
Negative reorder levels: 0
Invalid inventory values: 0
Inventory status mismatches: 0


In [0]:
inventory_silver = (
    inventory_bronze
    .withColumn("warehouse_id", F.upper(F.trim(F.col("warehouse_id"))))
    .withColumn("inventory_status", F.initcap(F.trim(F.col("inventory_status"))))
)

In [0]:
duplicate_inventory = (
    inventory_silver
    .groupBy("product_id", "warehouse_id")
    .count()
    .filter(F.col("count") > 1)
)

print("Duplicate product-warehouse combinations:", duplicate_inventory.count())

print("Total Silver inventory records:", inventory_silver.count())

print("Unique product-warehouse combinations:",
      inventory_silver.select("product_id", "warehouse_id").distinct().count())

Duplicate product-warehouse combinations: 0
Total Silver inventory records: 40000
Unique product-warehouse combinations: 40000


In [0]:
inventory_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.fashion_silver.inventory")

In [0]:
inventory_silver_saved = spark.table(
    "workspace.fashion_silver.inventory"
)

print("Silver inventory records:", inventory_silver_saved.count())

inventory_silver_saved.printSchema()

inventory_silver_saved.show(5, truncate=False)

Silver inventory records: 40000
root
 |-- product_id: integer (nullable = true)
 |-- warehouse_id: string (nullable = true)
 |-- stock_quantity: integer (nullable = true)
 |-- reorder_level: integer (nullable = true)
 |-- inventory_status: string (nullable = true)
 |-- price: double (nullable = true)
 |-- inventory_value: double (nullable = true)

+----------+------------+--------------+-------------+----------------+-------+---------------+
|product_id|warehouse_id|stock_quantity|reorder_level|inventory_status|price  |inventory_value|
+----------+------------+--------------+-------------+----------------+-------+---------------+
|0         |WH001       |254           |33           |In Stock        |4803.49|1220086.46     |
|1         |WH002       |14            |74           |Low Stock       |2405.49|33676.86       |
|2         |WH003       |217           |36           |In Stock        |2485.11|539268.87      |
|3         |WH004       |469           |97           |In Stock        |129